In [1]:
import scanpy as sc
import pandas as pd
import os

download raw counts and meta info from [GSE133345](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE133345).

# treat Monocyte as source cells

In [ ]:
count_df = pd.read_csv("raw/GSE133345_Quality_controled_UMI_data_of_all_1231_embryonic_cells.txt.gz", sep=' ', index_col=0).T
annot_df = pd.read_csv("raw/GSE133345_Annotations_of_all_1231_embryonic_cells_updated_0620.txt.gz", sep=' ', index_col=0)
adata = sc.AnnData(
    X=count_df.values,
    obs=annot_df.loc[count_df.index],
    var=pd.DataFrame(index=count_df.columns),
)
adata

AnnData object with n_obs × n_vars = 1231 × 26082
    obs: 'Site', 'Stage', 'cluster', 'UMAP1', 'UMAP2', 'nGene', 'nUMI'

In [5]:
adata.obs['perturbation'] = adata.obs['cluster'].apply(lambda x: 'control' if x == 'Monocyte' else 'unknown')

In [ ]:
adata.layers['raw'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=5000)
adata.write_h5ad('preprocessed.h5ad')

In [ ]:
split_df = adata.obs[['perturbation']].copy().assign(split='pred', subsplit='Monocyte')
split_df.drop(columns=['perturbation']).to_parquet('split_df_Monocyte.parquet')